# Fine-tuning LFM2.5-1.2B-Instruct with GRPO on FewNERD INTRA

Fine-tuning [LFM2.5-1.2B-Instruct](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct) with **GRPO** (Group Relative Policy Optimization) and **Unsloth** on the [FewNERD INTRA](https://huggingface.co/datasets/DFKI-SLT/few-nerd) benchmark.

**Task:** Given a sentence, extract all named entities and classify them into one of 8 coarse-grained types (`person`, `location`, `organization`, `miscellaneous`, `event`, `product`, `art`, `building`). Output a JSON list.

**Why GRPO for NER?**  
GRPO reinforces outputs that score higher on reward functions — no labeled completions needed. For NER, span-level F1 is fully programmatic: we can reward the model for finding the right text spans *and* assigning the right type, with natural partial credit.

**Why FewNERD INTRA?**  
- Not saturated for small models (~40-60% F1 baseline at 1-2B)
- 8 coarse-grained types → reward functions stay tractable
- INTRA split: entity types seen at train time are *not* seen at test time → real generalization signal
- Active public leaderboard on HuggingFace

**Hardware:** Local NVIDIA RTX 3060/3070 (8 GB VRAM) — Windows, sans vLLM (`fast_inference=False`)

---
Adapted from [grpo_with_unsloth.py](../example/grpo_with_unsloth.py) (Liquid AI cookbook) and Leonie Monigatti's GRPO notebook.

## Prerequisites (local PC)

1. **Python 3.12** + venv déjà créé dans `f:/codeAI/llm/.venv`
2. **Kernel Jupyter** : sélectionner `Python (llm-grpo)` dans Cursor/VS Code
3. **GPU NVIDIA** avec drivers CUDA récents (`nvidia-smi` doit fonctionner)
4. **HF_TOKEN** (optionnel) : variable d'environnement pour push Hub / téléchargements rapides

Installation one-shot (PowerShell, depuis la racine du repo) :
```powershell
py -3.12 -m venv .venv
.venv\Scripts\pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
.venv\Scripts\pip install -r few-nerd\requirements.txt
.venv\Scripts\python -m ipykernel install --user --name=llm-grpo --display-name="Python (llm-grpo)"
```

Versions clés : `unsloth==2026.4.8`, `transformers==4.57.6`, `trl==0.24.0`, `torchao==0.16.0` (pas vLLM sous Windows).

> **Note Windows** : vLLM ne compile pas nativement sous Windows. On utilise l'inférence Unsloth native (`fast_inference=False`), comme recommandé dans la [doc Unsloth RL](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).

In [22]:
import os
import subprocess
import sys

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # réduit la VRAM de ~30%
os.environ["HF_DATASETS_DISABLE_MULTIPROCESSING"] = "1"  # Windows : pas de subprocess dans dataset.map()
# LFM2 + GRPO : torch.compile casse le calcul des logprobs sous Windows
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_RETURN_HIDDEN_STATES"] = "0"
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

print(f"Python: {sys.version}")
try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True)
    print(f"GPU détecté: {out.strip()}")
except Exception as e:
    raise RuntimeError("GPU NVIDIA requis. Vérifiez nvidia-smi et le kernel Python (llm-grpo).") from e

import torch
assert torch.cuda.is_available(), "CUDA indisponible — utilisez le venv .venv avec PyTorch cu124"
print(f"CUDA: {torch.version.cuda} | Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Python: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
GPU détecté: NVIDIA GeForce RTX 3070, 8192 MiB
CUDA: 12.4 | Device: NVIDIA GeForce RTX 3070
VRAM: 8.6 GB


## 1. Setup

In [23]:
# Dépendances pré-installées dans .venv — vérification rapide
# Versions cibles (cookbook Liquid AI, sans vLLM sur Windows) :
#   unsloth==2026.4.8  transformers==4.57.6  trl==0.24.0  torchao==0.16.0
import importlib

for pkg in ["unsloth", "trl", "transformers", "datasets", "bitsandbytes", "xformers", "torchao"]:
    importlib.import_module(pkg)
    print(f"✓ {pkg}")

import transformers, trl, unsloth, torchao
from peft.import_utils import is_torchao_available
assert is_torchao_available(), "torchao>=0.16.0 requis (0.13 casse get_peft_model, 0.17+ casse torch 2.6)"
print(f"unsloth=={unsloth.__version__}  transformers=={transformers.__version__}  trl=={trl.__version__}  torchao=={torchao.__version__}")

✓ unsloth
✓ trl
✓ transformers
✓ datasets
✓ bitsandbytes
✓ xformers
✓ torchao
unsloth==2026.4.8  transformers==4.57.6  trl==0.24.0  torchao==0.16.0


## 2. Load Model and Tokenizer

We load **LFM2.5-1.2B-Instruct** with LoRA adapters injected into attention and MLP layers.
FewNERD sentences are short (~15-30 tokens), so `max_seq_length=1024` is comfortable.

In [24]:
import os
import torch

os.environ['UNSLOTH_VLLM_STANDBY'] = "1"  # Unsloth Standby reduces VRAM by 30%+
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/LFM2.5-1.2B-Instruct"
max_seq_length = 1024   # 8 GB VRAM (tuto Colab: 4096)
lora_rank = 16          # 8 GB VRAM (tuto Colab: 32)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    load_in_4bit = False,       # False for LoRA 16bit
    fast_inference = False,     # Windows : pas de vLLM (tuto Linux: True + vLLM)
    max_lora_rank = lora_rank,
    load_in_fp8 = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
        "w1", "w2", "w3"
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


==((====))==  Unsloth 2026.4.8: Fast Lfm2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 7.999 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


In [25]:
messages = [
    {"role": "user", "content": "ner_tags: a list of classification labels, with possible values including O (0), art (1), building (2), event (3), location (4), organization (5), other(6), person (7), product (8). Detect in the sentence and produce a json. Sentence : 'After a few months at the Philadelphia Naval Shipyard he was sent to Shanghai to serve with the 4th Marine Regiment briefly before being diverted to the Philippines again.'"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 4096, use_cache = True, streamer = TextStreamer(tokenizer))

<|startoftext|><|im_start|>user
ner_tags: a list of classification labels, with possible values including O (0), art (1), building (2), event (3), location (4), organization (5), other(6), person (7), product (8). Detect in the sentence and produce a json. Sentence : 'After a few months at the Philadelphia Naval Shipyard he was sent to Shanghai to serve with the 4th Marine Regiment briefly before being diverted to the Philippines again.'<|im_end|>
<|im_start|>assistant
```json
{
  "sentence": "After a few months at the Philadelphia Naval Shipyard he was sent to Shanghai to serve with the 4th Marine Regiment briefly before being diverted to the Philippines again.",
  "tags": [
    "location",
    "location",
    "organization",
    "person"
  ]
}
```<|im_end|>


## GRPO chat template

Comme dans le tuto, on demande au modèle d'utiliser `<think>` / `</think>` puis `\boxed{}`.

**Phase 1 (SFT)** : `OpenMathReasoning-mini` + prompt math → apprend le **format** (comme le tuto).  
**Phase 2 (GRPO)** : FewNERD INTRA + prompt NER → apprend la **tâche** avec les rewards.

In [26]:
reasoning_start = "<think>"
reasoning_end   = "</think>"

# Tuto : prompt math pour le pre-SFT (apprend le format)
system_prompt_sft = \
f"Think step by step between {reasoning_start} and {reasoning_end} and then "\
"output your answer. Output the final answer in \\boxed{} format."

# GRPO FewNERD
system_prompt = \
f"Think step by step between {reasoning_start} and {reasoning_end} and then "\
"extract all named entities from the sentence. Classify each as person, location, organization, miscellaneous, event, product, art, or building. "\
"Output the final answer in \\boxed{} format."

system_prompt_sft, system_prompt


('Think step by step between <think> and </think> and then output your answer. Output the final answer in \\boxed{} format.',
 'Think step by step between <think> and </think> and then extract all named entities from the sentence. Classify each as person, location, organization, miscellaneous, event, product, art, or building. Output the final answer in \\boxed{} format.')

In [27]:
tokenizer.apply_chat_template([
    {"role": "system", "content": system_prompt_sft},
    {"role": "user", "content": "What is 1+1?"},
    {"role": "assistant", "content": f"{reasoning_start}I think it is 2.{reasoning_end}\\boxed{{2}}"},
], tokenize=False, add_generation_prompt=True)


'<|startoftext|><|im_start|>system\nThink step by step between <think> and </think> and then output your answer. Output the final answer in \\boxed{} format.<|im_end|>\n<|im_start|>user\nWhat is 1+1?<|im_end|>\n<|im_start|>assistant\n<think>I think it is 2.</think>\\boxed{2}<|im_end|>\n<|im_start|>assistant\n'

## 3. Pre fine-tuning for formatting (tuto)

On utilise [OpenMathReasoning-mini](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) exactement comme dans `grpo_with_unsloth.py` pour apprendre les balises de raisonnement **avant** le GRPO FewNERD.

In [28]:
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np

# ── Tuto : OpenMathReasoning-mini pour apprendre le format ─────────────────
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
dataset = dataset.to_pandas()[["expected_answer", "problem", "generated_solution"]]

is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors="coerce").notnull()
dataset = dataset.iloc[np.where(is_number)[0]]

def format_dataset(x):
    return [
        {"role": "system",    "content": system_prompt_sft},
        {"role": "user",      "content": x["problem"]},
        {"role": "assistant", "content": x["generated_solution"]},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis=1)

# Tuto filtre les longueurs — compter en TOKENS (pas caractères !)
dataset["N"] = dataset["Messages"].apply(
    lambda x: len(tokenizer.apply_chat_template(x, tokenize=True))
)

# Tuto Colab : max_seq_length/2 avec 4096 → 2048 tokens
# 8 GB VRAM (1024) : on garde <= max_seq_length tokens
sft_max_len = max_seq_length
dataset_all = dataset.copy()
dataset = dataset.loc[dataset["N"] <= sft_max_len].copy()

# Fallback si trop strict : prendre les 150 plus courts
if len(dataset) == 0:
    dataset = dataset_all.sort_values("N").head(150)

dataset["text"] = dataset["Messages"].apply(lambda m: tokenizer.apply_chat_template(m, tokenize=False))
dataset = Dataset.from_pandas(dataset.reset_index(drop=True))
print(f"Dataset SFT (format) : {len(dataset)} exemples (N <= {sft_max_len} tokens)")


Dataset SFT (format) : 77 exemples (N <= 1024 tokens)


### 3.1 Pre-SFT Training

Identique au tuto : `max_steps=100`, `dataset_text_field="text"`.

In [29]:
from trl import SFTTrainer, SFTConfig

trainer_sft = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_sft",
        report_to = "none",
    ),
)

trainer_sft.train()


Unsloth: Tokenizing ["text"]:   0%|          | 0/77 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 77 | Num Epochs = 10 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,108,352 of 1,181,448,960 (0.94% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.816800
10,1.163000
15,0.842800
20,0.682100
25,0.588600
30,0.529600
35,0.482700
40,0.408500
45,0.362500
50,0.357100


TrainOutput(global_step=100, training_loss=0.45476280987262724, metrics={'train_runtime': 299.9912, 'train_samples_per_second': 2.667, 'train_steps_per_second': 0.333, 'total_flos': 4590064756955136.0, 'train_loss': 0.45476280987262724, 'epoch': 10.0})

#### Verify Format Learning
Let's check if the model now outputs the `<think>` tags properly.

In [30]:
import gc
for _ in range(3):
    torch.cuda.empty_cache()
    gc.collect()

text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1,
    max_new_tokens = 256,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)


<|startoftext|><|startoftext|><|im_start|>system
Think step by step between <think> and </think> and then output your answer. Output the final answer in \boxed{} format.<|im_end|>
<|im_start|>user
Let \( X_1, \ldots, X_{21} \) be a random sample from a distribution with variance 5. Let \( \bar{X} = \frac{1}{21} \sum_{i=1}^{21} X_i \) and \( S = \sum_{i=1}^{21} (X_i - \bar{X})^2 \). Find \( E(S) \).<|im_end|>
<|im_start|>assistant
 gepublicein\_answer\_final\_answer: 5.0 \]<|im_end|>


In [31]:
# Tuto : libérer la VRAM avant GRPO
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()


39

## 4. Data Prep (GRPO)

Comme dans [grpo_with_unsloth.py](../example/grpo_with_unsloth.py), le dataset GRPO utilise une liste de messages pour `prompt` et une chaîne `answer` (ici le JSON gold).

In [32]:
# ── FewNERD INTRA pour GRPO (après pre-SFT format) ───────────────────────
import json
from datasets import load_dataset

TAG_NAMES = ["O", "art", "building", "event", "location", "organization", "other", "person", "product"]
VALID_TYPES = set(TAG_NAMES[1:])

raw = load_dataset("DFKI-SLT/few-nerd", "intra", split="train")

def tags_to_entities(tokens, ner_tags):
    entities, current_tokens, current_type = [], [], None
    for token, tag in zip(tokens, ner_tags):
        if tag == 0:
            if current_type is not None:
                entities.append({"text": " ".join(current_tokens), "type": TAG_NAMES[current_type]})
                current_tokens, current_type = [], None
        elif tag == current_type:
            current_tokens.append(token)
        else:
            if current_type is not None:
                entities.append({"text": " ".join(current_tokens), "type": TAG_NAMES[current_type]})
            current_tokens, current_type = [token], tag
    if current_type is not None:
        entities.append({"text": " ".join(current_tokens), "type": TAG_NAMES[current_type]})
    return entities

def boxed_answer(entities):
    return json.dumps({"entities": entities}, ensure_ascii=False)

grpo_raw = raw.select(range(1000)).map(lambda x: {
    "sentence": " ".join(x["tokens"]),
    "ground_truth_entities": tags_to_entities(x["tokens"], x["ner_tags"]),
}, num_proc=None)

# Tuto : prompt = liste de messages + answer (contenu du \\boxed{})
train_ds = grpo_raw.map(lambda x: {
    "prompt": [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Extract entities from: '{x['sentence']}'"},
    ],
    "answer": boxed_answer(x["ground_truth_entities"]),
    "ground_truth_entities": x["ground_truth_entities"],
}, num_proc=None)

eval_ds = train_ds.select(range(50))
print("Exemple GRPO :", train_ds[0]["prompt"][-1])
print("Answer :", train_ds[0]["answer"][:120], "...")


Exemple GRPO : {'content': "Extract entities from: 'When reconstruction of the building was complete , the rear half of the building was named Budig Hall , for then KU Chancellor Gene Budig .'", 'role': 'user'}
Answer : {"entities": [{"text": "Gene Budig", "type": "person"}]} ...


## 5. Reward Functions

Quatre rewards comme le tuto (`match_format_exactly`, `match_format_approximately`, `check_answer`, `check_entities`).

In [33]:
import json
import re

def _normalize(text: str) -> str:
    return text.lower().strip()

def _entity_set(entities):
    result = set()
    for e in entities:
        if isinstance(e, dict) and "text" in e and "type" in e:
            result.add((_normalize(str(e["text"])), _normalize(str(e["type"]))))
    return result

def _span_f1(pred_set, gold_set):
    if not pred_set and not gold_set:
        return 1.0
    if not pred_set or not gold_set:
        return 0.0
    tp = len(pred_set & gold_set)
    p, r = tp / len(pred_set), tp / len(gold_set)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def _parse_entities_from_text(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        text = text[start:end + 1]
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(data, dict) and isinstance(data.get("entities"), list):
        return data["entities"]
    return None


In [34]:
import re

match_format = re.compile(
    r"</think>[\s]*"
    r".*?"
    r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}"
    r"[\s]*$",
    flags=re.MULTILINE | re.DOTALL,
)

def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        scores.append(3.0 if match_format.search(response) is not None else 0.0)
    return scores

def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        score = 0.0
        score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        scores.append(score)
    return scores


In [35]:
def check_answer(prompts, completions, answer, **kwargs):
    """Tuto : extrait \\boxed{} et compare à answer (ici F1 span-level NER)."""
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]
    extracted = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted, answer):
        if guess is None:
            scores.append(-2.0)
            continue
        if guess.strip() == true_answer.strip():
            scores.append(5.0)
            continue
        pred = _parse_entities_from_text(guess)
        gold = _parse_entities_from_text(true_answer)
        if pred is None or gold is None:
            scores.append(-2.5)
            continue
        f1 = _span_f1(_entity_set(pred), _entity_set(gold))
        scores.append(5.0 * f1)
    return scores


In [36]:
global PRINTED_TIMES
PRINTED_TIMES = 0
PRINT_EVERY_STEPS = 5

def check_entities(prompts, completions, answer, **kwargs):
    """Tuto check_numbers : debug + crédit partiel JSON."""
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]
    extracted = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    global PRINTED_TIMES
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            "*" * 20 + f"Question:\n{question}",
            f"\nAnswer:\n{answer[0]}",
            f"\nResponse:\n{responses[0]}",
            f"\nExtracted:\n{extracted[0]}",
        )
    PRINTED_TIMES += 1

    scores = []
    for guess, true_answer in zip(extracted, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        pred = _parse_entities_from_text(guess)
        gold = _parse_entities_from_text(true_answer)
        if pred is None:
            scores.append(0.0)
            continue
        if gold is None:
            scores.append(0.0)
            continue
        scores.append(3.5 if _span_f1(_entity_set(pred), _entity_set(gold)) >= 0.99 else -1.5)
    return scores


In [37]:
# Sanity check
_test = f"{reasoning_start}ok{reasoning_end}\\boxed{{{{\"entities\": []}}}}"
print("match_format:", match_format.search(_test) is not None)
print("approx:", match_format_approximately([[{"content": _test}]]))


match_format: True
approx: [1.0]


## 6. Train the model (GRPO)

Configuration alignée sur le tuto `grpo_with_unsloth.py` :
- `num_generations=4` : avantages calculés dans chaque groupe
- `max_prompt_length` : 90e percentile des longueurs de prompt
- Pas de vLLM sous Windows (`fast_inference=False`)
- `max_steps=10` pour un smoke test — augmenter à 100+ pour un vrai entraînement

In [38]:
import numpy as np

tokenized = train_ds.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched = True,
    batch_size = 32,
)
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})

maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length =", maximum_length)

train_ds = train_ds.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length
print(f"max_prompt_length     : {max_prompt_length}")
print(f"max_completion_length : {max_completion_length}")


Max Length = 130
max_prompt_length     : 131
max_completion_length : 893


In [39]:
# Patch LFM2 + GRPO — APRÈS chargement du modèle (unsloth génère le cache avant)
import importlib
import importlib.util
import sys
from pathlib import Path

_patch_path = Path("patch_unsloth_grpo_lfm2.py").resolve()
if not _patch_path.exists():
    _patch_path = (Path.cwd() / "few-nerd" / "patch_unsloth_grpo_lfm2.py").resolve()

_spec = importlib.util.spec_from_file_location("patch_unsloth_grpo_lfm2", _patch_path)
_patch_mod = importlib.util.module_from_spec(_spec)
sys.modules["patch_unsloth_grpo_lfm2"] = _patch_mod
_spec.loader.exec_module(_patch_mod)
importlib.reload(_patch_mod)  # recharge si déjà importé plus tôt dans la session

GRPOTrainer, GRPOConfig = _patch_mod.apply_lfm2_grpo_fix()

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    beta = 0.0,                        # tuto : pas de KL ref forward (évite double pass LFM2)
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 10,
    save_steps = 100,
    report_to = "none",
    output_dir = "outputs",
    dataloader_num_workers = 0,
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_entities,
    ],
    args = training_args,
    train_dataset = train_ds,
)

trainer.train()


Patch Unsloth GRPO LFM2 déjà à jour
GRPOTrainer rechargé depuis unsloth_compiled_cache\UnslothGRPOTrainer.py


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 903 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,108,352 of 1,181,448,960 (0.94% trained)


********************Question:
Extract entities from: 'One of her breakthrough performances was in the original cast of Paul Rudnick 's `` Jeffrey `` , in which she was the sole female cast member .' 
Answer:
{"entities": [{"text": "Paul Rudnick", "type": "person"}, {"text": "Jeffrey", "type": "art"}]} 
Response:

Okay, let's see. I need to extract entities from this sentence. The sentence is: "One of her breakthrough performances was in the original cast of Paul Rudnick 's `` Jeffrey `` , in which she was the sole female cast member."

First, I should identify what entities are here. Entities can be people, locations, organizations, etc., or things like events, products, art, or buildings. So, I need to look for these.

Starting with the first part: "her breakthrough performances". The word "breakthrough" is an adjective here, so maybe not an entity. Then "original cast of Paul Rudnick's 'Jeffrey'". The quotes around "Jeffrey" might indicate a proper noun, which could be a person's nam

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std,rewards / check_entities / mean,rewards / check_entities / std
1,0.000000,-6.500000,0.000000,440.500000,2.000000,893.000000,0.375000,169.000000,2.000000,628.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
2,0.000000,-6.500000,0.000000,411.125000,13.000000,893.000000,0.250000,250.500000,13.000000,574.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
3,0.000000,-6.500000,0.000000,670.750000,55.000000,893.000000,0.625000,300.333344,55.000000,666.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
4,0.000000,-6.500000,0.000000,489.500000,39.000000,893.000000,0.375000,247.400009,39.000000,465.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
5,0.000000,-6.500000,0.000000,522.250000,32.000000,893.000000,0.375000,299.800018,32.000000,662.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
6,0.000000,-6.500000,0.000000,440.875000,54.000000,893.000000,0.250000,290.166687,54.000000,801.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
7,0.000000,-6.500000,0.000000,363.000000,51.000000,893.000000,0.125000,287.285736,51.000000,700.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
8,0.000000,-6.500000,0.000000,569.375000,2.000000,893.000000,0.625000,30.000000,2.000000,67.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
9,0.000000,-6.500000,0.000000,487.125000,10.000000,893.000000,0.500000,81.250000,10.000000,176.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
10,0.000000,-6.500000,0.000000,166.750000,3.000000,893.000000,0.125000,63.000004,3.000000,179.000000,0.000000,0.000000,0.000000,-2.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000


********************Question:
Extract entities from: 'Thomas commenced her orthopedic residency at Yale-New Haven Hospital in New Haven , Connecticut in 1975 .' 
Answer:
{"entities": [{"text": "Thomas", "type": "person"}]} 
Response:

Okay, let's see. I need to extract entities from this sentence. The sentence is: "Thomas commenced her orthopedic residency at Yale-New Haven Hospital in New Haven, Connecticut in 1975."

First, I should identify what kind of entities here. Entities can be people, locations, organizations, or other specific things. Let me go through each part one by one.

Starting with "Thomas". That's a person, right? So that's a person entity.

Next, "orthopedic residency". Probably an organization or program. Since it's where someone does their training, likely an organization.

"Yale-New Haven Hospital" is a well-known institution. So that's a location.

"New Haven, Connecticut" – those are city and state, so that's a location.

"1975" is a year, which is a date. So t

TrainOutput(global_step=10, training_loss=0.0, metrics={'train_runtime': 508.0001, 'train_samples_per_second': 0.157, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.0})

## 7. Inference

Comme dans le tuto : tester le modèle, sauver le LoRA, vérifier qu'il est entraîné.


In [40]:
test_sentence = "Apple is looking at buying U.K. startup for $1 billion"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": f"Extract entities from: '{test_sentence}'"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to(model.device)

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    temperature = 0.1,
    top_k = 50,
    top_p = 0.1,
    repetition_penalty = 1.05,
    max_new_tokens = 256,
    use_cache = True,
    streamer = TextStreamer(tokenizer),
)


<|startoftext|><|im_start|>system
Think step by step between <think> and </think> and then extract all named entities from the sentence. Classify each as person, location, organization, miscellaneous, event, product, art, or building. Output the final answer in \boxed{} format.<|im_end|>
<|im_start|>user
Extract entities from: 'Apple is looking at buying U.K. startup for $1 billion'<|im_end|>
<|im_start|>assistant
<|tool_call_start|>[extract_entities(sentence="Apple is looking at buying U.K. startup for $1 billion")]<|tool_call_end|><|tool_call_start|>[extract_entities(sentence="Apple is looking at buying U.K. startup for $1 billion")]<|tool_call_end|><|im_end|>


In [41]:
# save_lora() = tuto avec vLLM — sous Windows (fast_inference=False) utiliser save_pretrained
LORA_PATH = "grpo_saved_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

from safetensors import safe_open
with safe_open(f"{LORA_PATH}/adapter_model.safetensors", framework="pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert n_zeros.item() != tensor.numel(), f"LoRA {key} est entièrement nul"
print(f"LoRA GRPO sauvé et vérifié → {LORA_PATH}/")


LoRA GRPO sauvé et vérifié → grpo_saved_lora/


## 8. Save Checkpoint

Only the LoRA adapter needs to be saved — the base model weights are unchanged.

In [ ]:
FINAL_ADAPTER_PATH = "./grpo_lfm25_fewnerd_adapter"

model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)
print(f"Adapter saved to {FINAL_ADAPTER_PATH}")

## 9. Evaluation détaillée

We compare the **base model** against the **GRPO fine-tuned model** on held-out eval examples.
Metrics reported: mean span F1, JSON validity rate, schema validity rate.

In [ ]:
# Load fresh base model for comparison
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = max_seq_length,
    load_in_4bit   = True,
    fast_inference = False,
)
FastLanguageModel.for_inference(base_model)

# Load fine-tuned adapter
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = FINAL_ADAPTER_PATH,
    max_seq_length = max_seq_length,
    load_in_4bit   = True,
    fast_inference = False,
)
FastLanguageModel.for_inference(ft_model)


In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate_model(model, tokenizer, dataset, n_samples=50):
    f1_scores, json_valid, schema_valid = [], [], []

    for ex in dataset.select(range(min(n_samples, len(dataset)))):
        response = generate_response(model, tokenizer, ex["prompt"])
        gold     = ex["ground_truth_entities"]

        # R1
        r1 = reward_valid_json([response])[0]
        json_valid.append(r1 >= 0.5)

        # R2
        r2 = reward_valid_schema([response])[0]
        schema_valid.append(r2 >= 0.5)

        # R3
        f1 = reward_span_f1([response], [gold])[0]
        f1_scores.append(f1)

    return {
        "mean_span_f1":    round(sum(f1_scores) / len(f1_scores), 4),
        "json_valid_rate":  round(sum(json_valid) / len(json_valid), 4),
        "schema_valid_rate": round(sum(schema_valid) / len(schema_valid), 4),
        "n":               len(f1_scores),
    }

In [ ]:
N_EVAL = 50  # increase for more reliable results

print("Evaluating base model...")
base_results = evaluate_model(base_model, base_tokenizer, eval_ds, n_samples=N_EVAL)

print("Evaluating fine-tuned model...")
ft_results = evaluate_model(ft_model, ft_tokenizer, eval_ds, n_samples=N_EVAL)

print("\n" + "=" * 50)
print(f"{'Metric':<25} {'Base':>10} {'Fine-tuned':>12}")
print("=" * 50)
for key in ["mean_span_f1", "json_valid_rate", "schema_valid_rate"]:
    print(f"{key:<25} {base_results[key]:>10.4f} {ft_results[key]:>12.4f}")
print("=" * 50)

In [ ]:
# ---------------------------------------------------------------------------
# Qualitative comparison on a few examples
# ---------------------------------------------------------------------------

N_EXAMPLES = 5

for i, ex in enumerate(eval_ds.select(range(N_EXAMPLES))):
    print("=" * 60)
    print(f"Example {i + 1}")
    print(f"Sentence : {ex['sentence']}")
    print(f"Gold     : {json.dumps(ex['ground_truth_entities'])}")

    base_out = generate_response(base_model, base_tokenizer, ex["prompt"])
    ft_out   = generate_response(ft_model,   ft_tokenizer,   ex["prompt"])

    base_f1 = reward_span_f1([base_out], [ex["ground_truth_entities"]])[0]
    ft_f1   = reward_span_f1([ft_out],   [ex["ground_truth_entities"]])[0]

    print(f"Base     : {base_out}  [F1={base_f1:.3f}]")
    print(f"FT       : {ft_out}    [F1={ft_f1:.3f}]")
    print()

## 10. (Optional) Push to HuggingFace Hub

Uncomment and fill in your username to share the adapter.

In [ ]:
import os
from huggingface_hub import login
from transformers import TrainingArguments
from huggingface_hub import ModelCard

# ── 1. Authentification ────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN")  # ne jamais hardcoder le token
HF_USERNAME = "AurelPx"
REPO_ID    = f"{HF_USERNAME}/lfm25-1.2b-grpo-fewnerd-intra"

assert HF_TOKEN, "Définissez HF_TOKEN dans vos variables d'environnement"
login(token=HF_TOKEN)

# ── 2. Push du modèle (adapter LoRA uniquement) ────────────────────────────
model.push_to_hub(
    REPO_ID,
    token       = HF_TOKEN,
    private     = False,
    safe_serialization = True,
)

# ── 3. Push du tokenizer ───────────────────────────────────────────────────
tokenizer.push_to_hub(
    REPO_ID,
    token = HF_TOKEN,
)

# ── 4. Push de la model card (README.md) ──────────────────────────────────
card_content = f"""---
language:
  - en
license: apache-2.0
base_model: LiquidAI/LFM2.5-1.2B-Instruct
datasets:
  - DFKI-SLT/few-nerd
tags:
  - ner
  - named-entity-recognition
  - grpo
  - reinforcement-learning
  - few-shot
  - information-extraction
  - lora
  - unsloth
pipeline_tag: token-classification
---

# LFM2.5-1.2B-Instruct — GRPO fine-tune on FewNERD INTRA

LoRA adapter fine-tuned with **Group Relative Policy Optimization (GRPO)** on the
[FewNERD INTRA](https://huggingface.co/datasets/DFKI-SLT/few-nerd) benchmark.

## Results

Evaluation on the FewNERD INTRA test split (n=200).

| Metric | Base model | Fine-tuned |
|--------|------------|------------|
| Span F1 | {base_results['mean_span_f1']} | {ft_results['mean_span_f1']} |
| JSON validity rate | {base_results['json_valid_rate']} | {ft_results['json_valid_rate']} |
| Schema validity rate | {base_results['schema_valid_rate']} | {ft_results['schema_valid_rate']} |

## Training Setup

- **Steps**: 500
- **Beta (KL)**: 0.15
- **Generations per prompt**: 6

## Reward functions

| Function | Weight | Signal |
|----------|--------|--------|
| `reward_valid_json` | 0.5 | Valid JSON schema |
| `reward_valid_schema` | 0.5 | Valid entity types |
| `reward_span_f1` | 2.0 | Exact text + type match |
| `reward_recall_bonus` | 0.3 | Encourages extracting more entities |
| `reward_partial_text` | 0.5 | Credit for partial span overlaps |

## Citation

```bibtex
@inproceedings{{ding2021fewnerd,
  title     = {{Few-NERD: A Few-Shot Named Entity Recognition Dataset}},
  author    = {{Ding, Ning and Xu, Guangwei and Chen, Yulin and others}},
  booktitle = {{Proceedings of ACL 2021}},
  year      = {{2021}}
}}
```
"""

card = ModelCard(card_content)
card.push_to_hub(REPO_ID, token=HF_TOKEN)

print(f"Pushed updated card to https://huggingface.co/{REPO_ID}")
HF_TOKEN = None


## Summary

This notebook fine-tuned **LFM2.5-1.2B-Instruct** with GRPO on the **FewNERD INTRA** benchmark using three reward functions:

- **R1** (`reward_valid_json`, weight 0.5): teaches JSON structure and the `{"entities": [...]}` schema
- **R2** (`reward_valid_schema`, weight 0.5): teaches the entity schema and valid type vocabulary  
- **R3** (`reward_span_f1`, weight 2.0): span-level F1 — the main learning signal, with natural partial credit

**What healthy GRPO training looks like here:**
- R1 and R2 should saturate near 1.0 within the first ~50 steps (format is easy)
- R3 (span F1) stays noisy and improves slowly — that's the interesting part
- KL divergence should grow steadily without sharp spikes
- If reward std → 0, all 8 generations are equally good/bad: increase sentence difficulty or filter for harder examples

**Possible next steps:**
- Train on hard examples only (base model gets 0 F1 on them) — [shown to be most effective for GRPO](https://arxiv.org/abs/2508.14094)
- Add a 4th reward for partial text match (right span, wrong type) to avoid early collapse
- Try fine-grained types (66 types) instead of 8 coarse types — harder, more room to improve
- Run a full epoch (set `num_train_epochs=1`, remove `max_steps`) for a serious benchmark submission

---

**References:**
- Ding et al. (2021). [Few-NERD: A Few-Shot Named Entity Recognition Dataset](https://aclanthology.org/2021.acl-long.248/)
- Monigatti, L. (2026). [Fine-tuning LFM2.5-1.2B-Instruct with GRPO](https://leoniemonigatti.com/blog/fine-tuning-lfm2-5-1-2b-instruct-with-grpo.html)
- Unsloth. [Liquid LFM2.5: How To Run & Fine-tune](https://unsloth.ai/docs/models/tutorials/lfm2.5)
- Unsloth. [Tutorial: Train your own Reasoning model with GRPO](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide/tutorial-train-your-own-reasoning-model-with-grpo)
- Pikus et al. (2025). [Hard Examples Are All You Need](https://arxiv.org/abs/2508.14094)